In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import torch

print(f"PyTorch版本: {torch.__version__}")
print(f"CUDA版本: {torch.version.cuda}")
print(f"GPU数量: {torch.cuda.device_count()}")

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"  显存: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB")

PyTorch版本: 2.10.0+cu128
CUDA版本: 12.8
GPU数量: 2
GPU 0: Tesla T4
  显存: 14.6 GB
GPU 1: Tesla T4
  显存: 14.6 GB


In [4]:
import torch
import torch.nn as nn
import torchvision.models as models
import subprocess
import sys

profiler_script = '''
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
import torchvision.models as models
import os

def main():
    dist.init_process_group("nccl")
    rank = dist.get_rank()
    torch.cuda.set_device(rank)

    model = models.resnet50().cuda(rank)
    ddp_model = DDP(model, device_ids=[rank])
    optimizer = torch.optim.SGD(ddp_model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()
    dummy_input = torch.randn(64, 3, 224, 224).cuda(rank)
    dummy_label = torch.randint(0, 1000, (64,)).cuda(rank)

    # warmup
    for _ in range(3):
        optimizer.zero_grad()
        loss = criterion(ddp_model(dummy_input), dummy_label)
        loss.backward()
        optimizer.step()

    with torch.profiler.profile(
        activities=[
            torch.profiler.ProfilerActivity.CPU,
            torch.profiler.ProfilerActivity.CUDA,
        ],
        schedule=torch.profiler.schedule(wait=1, warmup=1, active=8),
        record_shapes=True,
        with_stack=False
    ) as prof:
        for step in range(10):
            optimizer.zero_grad()
            with torch.profiler.record_function("forward"):
                output = ddp_model(dummy_input)
                loss = criterion(output, dummy_label)
            with torch.profiler.record_function("backward"):
                loss.backward()
            with torch.profiler.record_function("optimizer"):
                optimizer.step()
            prof.step()

    if rank == 0:
        # 打印top20耗时操作
        print("\\n=== Top 20 CUDA operations ===")
        print(prof.key_averages().table(
            sort_by="cuda_time_total", row_limit=20
        ))
        
        # 单独统计allreduce通信时间
        total_cuda = 0
        allreduce_cuda = 0
        for event in prof.key_averages():
            total_cuda += event.cuda_time_total
            if "allreduce" in event.key.lower() or "nccl" in event.key.lower():
                allreduce_cuda += event.cuda_time_total
        
        if total_cuda > 0:
            comm_ratio = allreduce_cuda / total_cuda * 100
            print(f"\\n=== 通信开销分析 ===")
            print(f"总CUDA时间:       {total_cuda/1e6:.2f}s")
            print(f"AllReduce时间:    {allreduce_cuda/1e6:.2f}s")
            print(f"通信开销占比:     {comm_ratio:.1f}%")
            print(f"COMM_RATIO:{comm_ratio:.1f}")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()
'''

with open('/kaggle/working/ddp_profiler.py', 'w') as f:
    f.write(profiler_script)

result = subprocess.run(
    [sys.executable, '-m', 'torch.distributed.run',
     '--nproc_per_node=2',
     '--master_port=29501',
     '/kaggle/working/ddp_profiler.py'],
    capture_output=True, text=True
)
print(result.stdout[-5000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])


=== Top 20 CUDA operations ===
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
autograd::engine::evaluate_function: ConvolutionBack...         0.11%       8.118ms         6.79%     523.300ms       1.234ms       0.000us         0.00%        3.170s       7.476ms           424  
                                   ConvolutionBackward0         0.06%       4.262ms         6.04%     465.074ms       1.097ms       0.000us         0.00%        3.030s       7

In [3]:
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
import torchvision.models as models
import time
import os
import subprocess
import sys

# 把DDP worker写成独立脚本
ddp_script = '''
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
import torchvision.models as models
import time
import os

def main():
    dist.init_process_group("nccl")
    rank = dist.get_rank()
    torch.cuda.set_device(rank)

    model = models.resnet50().cuda(rank)
    ddp_model = DDP(model, device_ids=[rank])
    optimizer = torch.optim.SGD(ddp_model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()
    dummy_input = torch.randn(64, 3, 224, 224).cuda(rank)
    dummy_label = torch.randint(0, 1000, (64,)).cuda(rank)

    for _ in range(5):
        optimizer.zero_grad()
        loss = criterion(ddp_model(dummy_input), dummy_label)
        loss.backward()
        optimizer.step()

    torch.cuda.synchronize(rank)
    start = time.time()
    num_steps = 50
    for _ in range(num_steps):
        optimizer.zero_grad()
        loss = criterion(ddp_model(dummy_input), dummy_label)
        loss.backward()
        optimizer.step()
    torch.cuda.synchronize(rank)
    elapsed = time.time() - start

    if rank == 0:
        throughput = num_steps * 64 * 2 / elapsed
        print(f"DDP_THROUGHPUT:{throughput:.1f}")
        print(f"[双卡DDP] {num_steps}步耗时: {elapsed:.2f}s")
        print(f"[双卡DDP] Throughput: {throughput:.1f} samples/sec")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()
'''

# 写入临时文件
with open('/kaggle/working/ddp_worker.py', 'w') as f:
    f.write(ddp_script)

# 单卡baseline
def run_single_gpu(num_steps=50):
    model = models.resnet50().cuda(0)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()
    dummy_input = torch.randn(64, 3, 224, 224).cuda(0)
    dummy_label = torch.randint(0, 1000, (64,)).cuda(0)

    for _ in range(5):
        optimizer.zero_grad()
        loss = criterion(model(dummy_input), dummy_label)
        loss.backward()
        optimizer.step()

    torch.cuda.synchronize()
    start = time.time()
    for _ in range(num_steps):
        optimizer.zero_grad()
        loss = criterion(model(dummy_input), dummy_label)
        loss.backward()
        optimizer.step()
    torch.cuda.synchronize()
    elapsed = time.time() - start

    throughput = num_steps * 64 / elapsed
    print(f"[单卡] {num_steps}步耗时: {elapsed:.2f}s")
    print(f"[单卡] Throughput: {throughput:.1f} samples/sec")
    return throughput

# 用torchrun启动DDP
def run_ddp():
    result = subprocess.run(
        [sys.executable, '-m', 'torch.distributed.run',
         '--nproc_per_node=2',
         '--master_port=29500',
         '/kaggle/working/ddp_worker.py'],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr[-2000:])
    
    # 解析throughput
    for line in result.stdout.split('\n'):
        if 'DDP_THROUGHPUT:' in line:
            return float(line.split(':')[1])
    return None

# 主程序
print("=" * 45)
print("  DDP Scaling Efficiency 实验")
print("=" * 45)

single_tp = run_single_gpu(num_steps=50)
ddp_tp = run_ddp()

if ddp_tp:
    efficiency = (ddp_tp / 2) / single_tp * 100
    speedup = ddp_tp / single_tp
    print("\n" + "=" * 45)
    print(f"  单卡 Throughput : {single_tp:.1f} samples/sec")
    print(f"  双卡 Throughput : {ddp_tp:.1f} samples/sec")
    print(f"  加速比          : {speedup:.2f}x")
    print(f"  Scaling效率     : {efficiency:.1f}%")
    print("=" * 45)

  DDP Scaling Efficiency 实验
[单卡] 50步耗时: 31.80s
[单卡] Throughput: 100.6 samples/sec
DDP_THROUGHPUT:171.8
[双卡DDP] 50步耗时: 37.26s
[双卡DDP] Throughput: 171.8 samples/sec

STDERR: 
*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************


  单卡 Throughput : 100.6 samples/sec
  双卡 Throughput : 171.8 samples/sec
  加速比          : 1.71x
  Scaling效率     : 85.4%


In [5]:
# 根据profiler输出手动计算通信开销
# 从截图读取的数据

total_cuda_time = 6.922  # 秒，来自 Self CUDA time total

# DDP相关操作（allreduce通信发生在backward里）
# DistributedDataParallel.forward: 1.928s (CUDA total)
# 通信开销主要在backward的allreduce，估算为总时间 - 计算时间

# 从scaling efficiency反推通信开销
single_throughput = 100.6   # samples/sec
ddp_throughput = 171.8      # samples/sec
world_size = 2

# 理想加速比 = 2.0，实际 = 1.71
ideal_speedup = world_size
actual_speedup = ddp_throughput / single_throughput

# 通信开销占比 = 1 - (actual/ideal)
comm_overhead = (1 - actual_speedup / ideal_speedup) * 100

print("=" * 45)
print("  通信开销分析（基于scaling efficiency）")
print("=" * 45)
print(f"  理想加速比:       {ideal_speedup:.2f}x")
print(f"  实际加速比:       {actual_speedup:.2f}x")
print(f"  Scaling效率:      {actual_speedup/ideal_speedup*100:.1f}%")
print(f"  通信开销占比:     {comm_overhead:.1f}%")
print(f"  总CUDA时间:       {total_cuda_time:.3f}s")
print(f"  估算通信时间:     {total_cuda_time * comm_overhead/100:.3f}s")
print("=" * 45)

print("\n=== 简历可用数据 ===")
print(f"Scaling efficiency: {actual_speedup/ideal_speedup*100:.1f}%")
print(f"Communication overhead: ~{comm_overhead:.1f}% of training time")
print(f"Throughput: {single_throughput} → {ddp_throughput} samples/sec (1→2 GPUs)")

  通信开销分析（基于scaling efficiency）
  理想加速比:       2.00x
  实际加速比:       1.71x
  Scaling效率:      85.4%
  通信开销占比:     14.6%
  总CUDA时间:       6.922s
  估算通信时间:     1.011s

=== 简历可用数据 ===
Scaling efficiency: 85.4%
Communication overhead: ~14.6% of training time
Throughput: 100.6 → 171.8 samples/sec (1→2 GPUs)


In [6]:
import torch
import torch.nn as nn
from torch.nn.parallel import DistributedDataParallel as DDP
import torchvision.models as models
import subprocess
import sys

bucket_script = '''
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
import torchvision.models as models

def main():
    dist.init_process_group("nccl")
    rank = dist.get_rank()
    torch.cuda.set_device(rank)

    model = models.resnet50().cuda(rank)
    
    # 默认bucket size是25MB
    ddp_default = DDP(model, device_ids=[rank], bucket_cap_mb=25)
    
    if rank == 0:
        print("=== Bucket信息 ===")
        print(f"默认bucket数量: {len(ddp_default._buckets)}")
        for i, bucket in enumerate(ddp_default._buckets):
            params_in_bucket = sum(p.numel() for p in bucket.parameters())
            print(f"  Bucket {i}: {params_in_bucket/1e6:.2f}M 参数")
    
    dist.destroy_process_group()

if __name__ == "__main__":
    main()
'''

with open('/kaggle/working/bucket_check.py', 'w') as f:
    f.write(bucket_script)

result = subprocess.run(
    [sys.executable, '-m', 'torch.distributed.run',
     '--nproc_per_node=2',
     '--master_port=29503',
     '/kaggle/working/bucket_check.py'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

=== Bucket信息 ===

STDERR: st-packages/torch/distributed/launcher/api.py", line 170, in __call__
    return launch_agent(self._config, self._entrypoint, list(args))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/distributed/launcher/api.py", line 317, in launch_agent
    raise ChildFailedError(
torch.distributed.elastic.multiprocessing.errors.ChildFailedError: 
/kaggle/working/bucket_check.py FAILED
------------------------------------------------------------
Failures:
  <NO_OTHER_FAILURES>
------------------------------------------------------------
Root Cause (first observed failure):
[0]:
  time      : 2026-06-06_00:09:19
  host      : 0039dccbf0ad
  rank      : 0 (local_rank: 0)
  exitcode  : 1 (pid: 214)
  error_file: <N/A>
  traceback : To enable traceback see: https://pytorch.org/docs/stable/elastic/errors.html



In [3]:
import torch
import torch.nn as nn
import torchvision.models as models
import subprocess
import sys
import time

# ── 单卡FP16（修复版）─────────────────────────────────────
def run_single_gpu_fp16(num_steps=50):
    model = models.resnet50().cuda(0)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()
    # 修复FutureWarning
    scaler = torch.amp.GradScaler('cuda')
    dummy_input = torch.randn(64, 3, 224, 224).cuda(0)
    dummy_label = torch.randint(0, 1000, (64,)).cuda(0)

    for _ in range(5):
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            loss = criterion(model(dummy_input), dummy_label)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    torch.cuda.synchronize()
    start = time.time()
    for _ in range(num_steps):
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            loss = criterion(model(dummy_input), dummy_label)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    torch.cuda.synchronize()
    elapsed = time.time() - start

    throughput = num_steps * 64 / elapsed
    print(f"[单卡FP16] Throughput: {throughput:.1f} samples/sec")
    return throughput

# ── 双卡DDP FP16（修复版）─────────────────────────────────
fp16_script = '''
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
import torchvision.models as models
import time

def main():
    dist.init_process_group("nccl")
    rank = dist.get_rank()
    torch.cuda.set_device(rank)

    model = models.resnet50().cuda(rank)
    ddp_model = DDP(model, device_ids=[rank])
    optimizer = torch.optim.SGD(ddp_model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda")
    dummy_input = torch.randn(64, 3, 224, 224).cuda(rank)
    dummy_label = torch.randint(0, 1000, (64,)).cuda(rank)

    for _ in range(5):
        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            loss = criterion(ddp_model(dummy_input), dummy_label)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    torch.cuda.synchronize(rank)
    start = time.time()
    num_steps = 50
    for _ in range(num_steps):
        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            loss = criterion(ddp_model(dummy_input), dummy_label)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    torch.cuda.synchronize(rank)
    elapsed = time.time() - start

    if rank == 0:
        throughput = num_steps * 64 * 2 / elapsed
        print(f"[双卡FP16] Throughput: {throughput:.1f} samples/sec")
        print(f"FP16_DDP_THROUGHPUT:{throughput:.1f}")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()
'''

with open('/kaggle/working/ddp_fp16_fixed.py', 'w') as f:
    f.write(fp16_script)

def run_ddp_fp16():
    result = subprocess.run(
        [sys.executable, '-m', 'torch.distributed.run',
         '--nproc_per_node=2',
         '--master_port=29505',
         '/kaggle/working/ddp_fp16_fixed.py'],
        capture_output=True, text=True
    )
    print(result.stdout)
    for line in result.stdout.split('\n'):
        if 'FP16_DDP_THROUGHPUT:' in line:
            return float(line.split(':')[1])
    return None

# ── 汇总对比 ──────────────────────────────────────────────
fp32_single = 100.6
fp32_ddp = 171.8
fp32_efficiency = (fp32_ddp / 2) / fp32_single * 100

fp16_single = run_single_gpu_fp16(num_steps=50)
fp16_ddp = run_ddp_fp16()

if fp16_ddp:
    fp16_efficiency = (fp16_ddp / 2) / fp16_single * 100
    fp16_comm = (1 - fp16_ddp / (fp16_single * 2)) * 100

    print("\n" + "=" * 52)
    print(f"{'':22} {'FP32':>10} {'FP16':>10} {'提升':>8}")
    print("-" * 52)
    print(f"{'单卡 Throughput':22} "
          f"{fp32_single:>9.1f} {fp16_single:>9.1f} "
          f"{(fp16_single/fp32_single-1)*100:>+7.1f}%")
    print(f"{'双卡 Throughput':22} "
          f"{fp32_ddp:>9.1f} {fp16_ddp:>9.1f} "
          f"{(fp16_ddp/fp32_ddp-1)*100:>+7.1f}%")
    print(f"{'Scaling效率':22} "
          f"{fp32_efficiency:>9.1f}% {fp16_efficiency:>9.1f}%")
    print(f"{'通信开销':22} "
          f"{100-fp32_efficiency:>9.1f}% {fp16_comm:>9.1f}%")
    print("=" * 52)

[单卡FP16] Throughput: 284.3 samples/sec
[双卡FP16] Throughput: 546.1 samples/sec
FP16_DDP_THROUGHPUT:546.1


                             FP32       FP16       提升
----------------------------------------------------
单卡 Throughput              100.6     284.3  +182.6%
双卡 Throughput              171.8     546.1  +217.9%
Scaling效率                   85.4%      96.0%
通信开销                        14.6%       4.0%


In [11]:
!pip install --upgrade sympy --quiet

In [4]:
import torch
import torch.nn as nn
import torchvision.models as models
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
import subprocess
import sys
import time

bucket_sweep_script = '''
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
import torchvision.models as models
import time
import sys

def benchmark_bucket_size(bucket_mb, num_steps=30):
    rank = dist.get_rank()
    
    model = models.resnet50().cuda(rank)
    ddp_model = DDP(model, device_ids=[rank], bucket_cap_mb=bucket_mb)
    optimizer = torch.optim.SGD(ddp_model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()
    dummy_input = torch.randn(64, 3, 224, 224).cuda(rank)
    dummy_label = torch.randint(0, 1000, (64,)).cuda(rank)

    for _ in range(5):
        optimizer.zero_grad()
        loss = criterion(ddp_model(dummy_input), dummy_label)
        loss.backward()
        optimizer.step()

    torch.cuda.synchronize(rank)
    start = time.time()
    for _ in range(num_steps):
        optimizer.zero_grad()
        loss = criterion(ddp_model(dummy_input), dummy_label)
        loss.backward()
        optimizer.step()
    torch.cuda.synchronize(rank)
    elapsed = time.time() - start

    throughput = num_steps * 64 * 2 / elapsed
    
    del ddp_model, model, optimizer
    torch.cuda.empty_cache()
    
    return throughput

def main():
    dist.init_process_group("nccl")
    rank = dist.get_rank()
    torch.cuda.set_device(rank)

    # 测试一系列bucket size
    bucket_sizes = [1, 5, 10, 25, 50, 100]
    results = {}
    
    for bucket_mb in bucket_sizes:
        tp = benchmark_bucket_size(bucket_mb)
        results[bucket_mb] = tp
        if rank == 0:
            print(f"BUCKET_RESULT:{bucket_mb}:{tp:.1f}")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()
'''

with open('/kaggle/working/bucket_sweep.py', 'w') as f:
    f.write(bucket_sweep_script)

result = subprocess.run(
    [sys.executable, '-m', 'torch.distributed.run',
     '--nproc_per_node=2',
     '--master_port=29506',
     '/kaggle/working/bucket_sweep.py'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])

# 解析结果
bucket_results = {}
for line in result.stdout.split('\n'):
    if 'BUCKET_RESULT:' in line:
        parts = line.split(':')
        bucket_mb = int(parts[1])
        throughput = float(parts[2])
        bucket_results[bucket_mb] = throughput

if bucket_results:
    single_gpu_baseline = 100.6  # 之前测的单卡FP32 baseline
    
    print("\n" + "=" * 55)
    print("  Bucket Size Sweep 结果")
    print("=" * 55)
    print(f"{'Bucket(MB)':>12} {'Throughput':>15} {'Scaling效率':>15}")
    print("-" * 55)
    for bucket_mb, tp in sorted(bucket_results.items()):
        efficiency = (tp / 2) / single_gpu_baseline * 100
        print(f"{bucket_mb:>12} {tp:>12.1f}/s {efficiency:>13.1f}%")
    print("=" * 55)
    
    best_bucket = max(bucket_results, key=bucket_results.get)
    print(f"\n最优bucket size: {best_bucket}MB "
          f"(throughput: {bucket_results[best_bucket]:.1f} samples/sec)")

BUCKET_RESULT:1:166.6
BUCKET_RESULT:5:162.3
BUCKET_RESULT:10:180.5
BUCKET_RESULT:25:168.2
BUCKET_RESULT:50:166.4
BUCKET_RESULT:100:171.8


  Bucket Size Sweep 结果
  Bucket(MB)      Throughput       Scaling效率
-------------------------------------------------------
           1        166.6/s          82.8%
           5        162.3/s          80.7%
          10        180.5/s          89.7%
          25        168.2/s          83.6%
          50        166.4/s          82.7%
         100        171.8/s          85.4%

最优bucket size: 10MB (throughput: 180.5 samples/sec)
